# DT401 Week 3 Section 2: Data Management & FAIR Principles

**Module:** Data Fundamentals  
**Week:** 3 of 7  
**Section:** 2 of 3  
**Estimated Time:** 45 minutes

---

## Learning Objective

Understand data lifecycle management and apply FAIR principles to make data findable, accessible, interoperable, and reusable.

By the end of this notebook, you will be able to:
- Identify data lifecycle stages and appropriate management practices for each
- Apply FAIR principles to assess data management quality
- Evaluate metadata quality and completeness
- Design archiving strategies based on business requirements

---

## Context Recap

In Section 1, you learned to assess data quality across six dimensions. Quality assessment answers "is this data fit for purpose right now?" Data management answers "how do we ensure data remains fit for purpose throughout its lifetime?" This section explores lifecycle stages, archiving strategies, and FAIR principles that govern professional data stewardship.

---

## How to Use This Notebook

This notebook contains a mix of instructional text, Python code cells, and interactive exercises. Follow the instructions in each section, run the code cells to see outputs, and complete the exercises to reinforce your learning.

You will find several types of boxes throughout the notebook to guide your learning:

<div class="alert alert-block alert-warning">
<b>Reminder boxes:</b> <br>
These boxes contain important reminders about how this section relates to the overall module and your learning journey.
</div>

<div class="alert alert-block alert-info">
<b>Reflection boxes:</b> <br>
These have pointers as to what we should be looking for or comments on how to approach the topic discussed.
</div>

<div class="alert alert-block alert-success">
<b>Task boxes:</b> <br>
These have questions or tasks you should complete before progressing.

These may include hidden hints or solutions.
<details>
    <summary style="color:black;font-weight:bold">Click here for a hint</summary>

Hints will guide you toward the solution without giving away the answer completely.

</details>

</div>

---

## Part 1: Understanding Data Lifecycle Stages

### Visual Overview

```mermaid
flowchart LR
    A[Creation] --> B[Storage]
    B --> C[Usage]
    C --> D[Archiving]
    D --> E[Deletion]
    
    C -->|Active Data| C
    D -->|Compliance Hold| D
```

**What this diagram shows:**  
Data moves through five lifecycle stages. Active data cycles between storage and usage; archived data is retained for compliance or historical reference before eventual deletion.

**Why this matters in your role:**  
Understanding lifecycle stages helps you decide when to archive old data (improving system performance), when to delete data (meeting GDPR requirements), and how long to retain data (satisfying compliance). Poor lifecycle management leads to slow systems, compliance failures, and excessive storage costs.

---

### Understanding Lifecycle Stages

Each lifecycle stage has specific management requirements:

**1. Creation**: Data enters the organisation through sensors, user input, APIs, or external sources. Focus: validation, quality checks, proper formatting.

**2. Storage**: Data is kept in databases, data warehouses, or file systems. Focus: security, backup, access control, efficient retrieval.

**3. Usage**: Active deployment for analysis, reporting, decision-making. Focus: availability, performance, documentation, quality maintenance.

**4. Archiving**: Removal from active systems while retaining for compliance or history. Focus: cost-effective storage, preservation, retrieval mechanisms.

**5. Deletion**: Permanent removal when no longer needed. Focus: compliance with retention policies, secure erasure, audit trails.

<div class="alert alert-block alert-warning">
<b>Reminder:</b> <br>
Lifecycle management isn't one-size-fits-all. Financial transaction data may need 7-year retention; personal browsing data might require 12-month deletion under GDPR. Always align lifecycle policies with legal, regulatory, and business requirements.
</div>

### Worked Example: Analyzing Data Age Distribution

Let's examine our weather dataset to understand its age distribution - a key factor in lifecycle management decisions.

In [ ]:
import pandas as pd
from datetime import datetime

# Load weather data
df = pd.read_csv('../open-meteo-51.49N0.16W23m.csv', skiprows=3, low_memory=False)
df = df[df['time'].str.match(r'^\d{4}-\d{2}-\d{2}T', na=False)].reset_index(drop=True)
for col in df.columns[1:]:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Convert time column to datetime
df['time'] = pd.to_datetime(df['time'])

# Calculate data age
latest_date = df['time'].max()
earliest_date = df['time'].min()

print(f"Dataset time range:")
print(f"  Earliest: {earliest_date}")
print(f"  Latest: {latest_date}")
print(f"  Span: {(latest_date - earliest_date).days} days")

<div class="alert alert-block alert-info">
<b>What just happened?</b><br>

- We converted the 'time' column from text to datetime objects for date calculations
- We found the earliest and latest timestamps in the dataset
- We calculated the total time span covered by the data
- This information helps determine archiving strategies (e.g., archive data >2 years old)

**Key lesson:** Understanding data age distribution informs lifecycle decisions. Recent data stays active; older data becomes archiving candidates.

</div>

Now analyze how much data falls into different age categories:

In [ ]:
# Calculate age from latest date
df['age_days'] = (latest_date - df['time']).dt.days

# Categorize by age
recent = len(df[df['age_days'] <= 30])  # Last 30 days
active = len(df[(df['age_days'] > 30) & (df['age_days'] <= 365)])  # 1 month to 1 year
archive_candidate = len(df[df['age_days'] > 365])  # Older than 1 year

print(f"\nData distribution by age:")
print(f"  Recent (≤30 days): {recent:,} records ({recent/len(df)*100:.1f}%)")
print(f"  Active (30d-1y): {active:,} records ({active/len(df)*100:.1f}%)")
print(f"  Archive candidates (>1y): {archive_candidate:,} records ({archive_candidate/len(df)*100:.1f}%)")

---

### Modified Task: Calculate Storage Savings from Archiving

<div class="alert alert-block alert-success">
<b>Your turn:</b> <br>
Calculate potential storage savings if we archive data older than 2 years. Assume archive storage costs 10% of active storage.

<details>
    <summary style="color:black;font-weight:bold">Click here for a hint</summary>

Steps:
1. Count records older than 2 years (730 days)
2. Calculate percentage of total dataset
3. If 1000 records cost £100 in active storage, same records cost £10 in archive storage
4. Savings = (active cost - archive cost) for the old data

</details>

</div>

<details>
    <summary style="color:green;font-weight:bold">Click here for the solution</summary>

```python
# Find records older than 2 years
archive_2y = len(df[df['age_days'] > 730])
archive_pct = (archive_2y / len(df)) * 100

# Calculate storage savings (assuming £1 per 1000 records active storage)
active_cost = (archive_2y / 1000) * 1.00  # £1 per 1000 records
archive_cost = active_cost * 0.10  # Archive costs 10% of active
savings = active_cost - archive_cost

print(f"\nArchiving Strategy (2-year threshold):")
print(f"  Records to archive: {archive_2y:,} ({archive_pct:.1f}% of dataset)")
print(f"  Current active storage cost: £{active_cost:.2f}")
print(f"  Archive storage cost: £{archive_cost:.2f}")
print(f"  Annual savings: £{savings:.2f} ({(savings/active_cost)*100:.0f}% reduction)")
```

</details>

In [ ]:
# Your code here
# Calculate archiving savings for 2-year threshold



**Expected output:** Count of records to archive, cost comparison, and percentage savings.

**Success check:** Archiving should show significant cost savings (typically 80-90% reduction for archived data). This demonstrates the business value of lifecycle management.

---

## Part 2: FAIR Principles - Findability

### Visual Overview

```mermaid
flowchart TD
    A[Findable Data] --> B[Unique Identifier]
    A --> C[Rich Metadata]
    A --> D[Searchable Repository]
    
    B --> E[Data Discovery]
    C --> E
    D --> E
```

**What this diagram shows:**  
Findable data requires three elements: unique identifiers, rich metadata, and searchable storage.

**Why this matters in your role:**  
If colleagues can't find data, they either recreate it (wasting effort) or make decisions without it (missing insights). Findability saves time and improves decision quality.

---

### Understanding Findability

Findability ensures data can be discovered by humans and machines. Key components:

**Unique Identifiers**: Each dataset has a persistent, globally unique identifier (like DOI for academic papers, or UUIDs for internal datasets).

**Rich Metadata**: Descriptive information about the data:
- What: Description of dataset content
- Who: Creator, contact person, organisation
- When: Creation date, update frequency
- Where: Geographic coverage, storage location
- How: Collection methodology, processing steps

**Searchable Repository**: Data catalogued in systems that support search and discovery.

<div class="alert alert-block alert-warning">
<b>Reminder:</b> <br>
Poor findability is common. Teams often can't locate data that exists within their own organisation, leading to duplicate data collection efforts and missed opportunities for data reuse.
</div>

### Worked Example: Examining Dataset Metadata

Let's examine the metadata available for our weather dataset:

In [ ]:
# Extract basic metadata
metadata = {
    'Dataset Name': 'London Weather Observations',
    'Source': 'Open-Meteo API',
    'Location': 'London, UK (51.49°N, 0.16°W)',
    'Time Range': f"{earliest_date} to {latest_date}",
    'Variables': len(df.columns),
    'Observations': len(df),
    'Update Frequency': 'Hourly'
}

print("DATASET METADATA")
for key, value in metadata.items():
    print(f"  {key}: {value}")

<div class="alert alert-block alert-info">
<b>What just happened?</b><br>

- We created a metadata dictionary describing the dataset
- We included key information: what, where, when, how many
- This metadata helps users understand if the dataset meets their needs
- Good metadata answers "is this the data I'm looking for?" without examining all the data

**Key lesson:** Metadata should be comprehensive enough to assess data suitability without downloading or processing it.

</div>

Now create a detailed variable catalog:

In [ ]:
# Create variable metadata
print("\nVARIABLE CATALOG")
print(f"Total variables: {len(df.columns)}\n")

# Sample key variables with descriptions
key_vars = {
    'temperature_2m (°C)': 'Air temperature at 2 meters height',
    'relative_humidity_2m (%)': 'Relative humidity at 2 meters',
    'wind_speed_10m (mp/h)': 'Wind speed at 10 meters height',
    'precipitation (mm)': 'Total precipitation amount'
}

for var, description in key_vars.items():
    if var in df.columns:
        print(f"  {var}")
        print(f"    Description: {description}")
        print(f"    Range: {df[var].min():.1f} to {df[var].max():.1f}")
        print(f"    Missing: {df[var].isnull().sum()} values\n")

---

### Modified Task: Assess Metadata Completeness

<div class="alert alert-block alert-success">
<b>Your turn:</b> <br>

Evaluate metadata completeness using these criteria:
- Dataset identifier: Present/Missing
- Creator information: Present/Missing
- Time coverage: Present/Missing
- Geographic coverage: Present/Missing
- Variable descriptions: Present/Missing
- Access conditions: Present/Missing

Calculate completeness score (percentage of criteria met).

<details>
    <summary style="color:black;font-weight:bold">Click here for a hint</summary>

Create a checklist dictionary with True/False for each criterion. Count how many are True, divide by total criteria, multiply by 100 for percentage.

</details>

</div>

<details>
    <summary style="color:green;font-weight:bold">Click here for the solution</summary>
<br>

| Criterion | Metadata Key | Status |
|-----------|--------------|--------|
| Dataset identifier | Dataset Name | Present |
| Creator information | Source | Present |
| Time coverage | Time Range | Present |
| Geographic coverage | Location | Present |
| Variable descriptions | Variables | Present |
| Access conditions | - | Missing |

- Total criteria: 6
- Criteria met: 5
- Completeness score: (5/6) * 100 = 83%

Here is a version of the code that could be used to somewhat automate this assessment:

```python
# Metadata completeness checklist
metadata_checklist = {
    'Dataset identifier': 'Open-Meteo API London Weather' in str(metadata),  # True
    'Creator information': 'Open-Meteo API' in str(metadata),  # True
    'Time coverage': earliest_date is not None and latest_date is not None,  # True
    'Geographic coverage': 'London, UK' in str(metadata),  # True
    'Variable descriptions': len(key_vars) > 0,  # True
    'Access conditions': False  # Not documented (assumed open access)
}

print("METADATA COMPLETENESS ASSESSMENT")
for criterion, present in metadata_checklist.items():
    status = "✓ Present" if present else "✗ Missing"
    print(f"  {criterion}: {status}")

completeness_score = (sum(metadata_checklist.values()) / len(metadata_checklist)) * 100
print(f"\nMetadata Completeness: {completeness_score:.0f}%")

if completeness_score >= 80:
    print("✓ GOOD: Metadata supports data discovery")
elif completeness_score >= 60:
    print("⚠ FAIR: Metadata is adequate but could be improved")
else:
    print("✗ POOR: Metadata is insufficient for discovery")
```

</details>

In [ ]:
# Your code here
# Assess metadata completeness



**Expected output:** Checklist showing which metadata elements are present, plus completeness percentage.

**Success check:** Good metadata typically scores 80%+. Scores below 60% indicate discovery challenges.

---

## Part 3: FAIR Principles - Accessibility

### Visual Overview

```mermaid
flowchart LR
    A[Accessible Data] --> B[Standard Protocols]
    A --> C[Authentication]
    A --> D[Metadata Persistence]
    
    B --> E[Data Retrieval]
    C --> E
    D --> E
```

**What this diagram shows:**  
Accessible data can be retrieved via standard protocols with appropriate authentication, and metadata remains available even when data is removed.

**Why this matters in your role:**  
Accessible doesn't mean "open to everyone" - it means "retrievable by authorised users through known methods." Patient data is accessible to clinicians (with authorisation) but not the public. Financial data is accessible to auditors but not competitors.

---

### Understanding Accessibility

Accessibility ensures authorised users can retrieve data when needed. Key aspects:

**Standard Protocols**: Data retrievable via well-documented methods (HTTP, FTP, API, SQL queries) not proprietary or undocumented systems.

**Authentication & Authorisation**: Clear access control - who can access what data, when, and how.

**Metadata Persistence**: Even if data is deleted, metadata remains describing what existed and why it was removed.

**Format Accessibility**: Data in formats that don't require expensive proprietary software.

<div class="alert alert-block alert-warning">
<b>Reminder:</b> <br>
Accessibility differs from openness. Open data is publicly accessible without restrictions. Accessible data can be retrieved by authorised users - access may be restricted to protect privacy, security, or commercial interests.
</div>

### Worked Example: Assessing Data Format Accessibility

Let's evaluate whether our data uses accessible formats:

In [ ]:
# Assess format accessibility
format_assessment = {
    'File Format': 'CSV (Comma-Separated Values)',
    'Open Standard': True,
    'Human Readable': True,
    'Machine Readable': True,
    'Proprietary Software Required': False,
    'Common Tools': 'Excel, Python pandas, R, any text editor'
}

print("FORMAT ACCESSIBILITY ASSESSMENT")
for criterion, value in format_assessment.items():
    print(f"  {criterion}: {value}")

<div class="alert alert-block alert-info">
<b>What just happened?</b><br>

- We assessed whether the data format supports accessibility
- CSV is an open standard readable by many tools
- No proprietary software needed (unlike some Excel features or SPSS files)
- Both humans (text editors) and machines (pandas) can read it

**Key lesson:** Open formats like CSV, JSON, XML promote accessibility. Proprietary formats (some .xlsx features, SPSS .sav files) create barriers.

</div>

---

### Modified Task: Design Access Control Policy

<div class="alert alert-block alert-success">
<b>Your turn:</b> <br>
Design an access control policy for weather data in different contexts:

1. Public weather forecasting: Who should access? What restrictions?
2. Commercial agriculture: Who should access? What restrictions?
3. Infrastructure planning: Who should access? What restrictions?

For each, specify: access level (public/restricted), required authentication, usage restrictions.

<details>
    <summary style="color:black;font-weight:bold">Click here for a hint</summary>

Consider:
- Public forecasting benefits from open access
- Commercial users might pay for API access
- Infrastructure planning needs authorised government access

Create a dictionary or table showing access requirements for each use case.

</details>

</div>

<details>
    <summary style="color:green;font-weight:bold">Click here for the solution</summary>

```python
# Access control policy design
access_policy = {
    'Public Weather Forecasting': {
        'Access Level': 'Public',
        'Authentication': 'None required',
        'Usage Restrictions': 'Attribution required',
        'Rationale': 'Public safety benefit from open access'
    },
    'Commercial Agriculture': {
        'Access Level': 'Restricted (paid)',
        'Authentication': 'API key required',
        'Usage Restrictions': 'Commercial license, rate limits apply',
        'Rationale': 'High-value commercial use justifies licensing'
    },
    'Infrastructure Planning': {
        'Access Level': 'Restricted (government)',
        'Authentication': 'Government credentials required',
        'Usage Restrictions': 'Official use only, data sharing agreement',
        'Rationale': 'Sensitive infrastructure data requires authorisation'
    }
}

print("ACCESS CONTROL POLICY\n")
for use_case, policy in access_policy.items():
    print(f"{use_case}:")
    for key, value in policy.items():
        print(f"  {key}: {value}")
    print()
```

</details>

In [ ]:
# Your code here
# Design access control policies for different use cases



**Expected output:** Policy table showing access requirements for three use cases.

**Success check:** Policies should balance accessibility with appropriate restrictions based on use case sensitivity and commercial value.

---

## Part 4: FAIR Principles - Interoperability

### Visual Overview

```mermaid
flowchart TD
    A[Interoperable Data] --> B[Standard Formats]
    A --> C[Common Vocabularies]
    A --> D[Documented Schemas]
    
    B --> E[Data Integration]
    C --> E
    D --> E
```

**What this diagram shows:**  
Interoperable data uses standard formats, common vocabularies, and documented schemas enabling integration with other datasets.

**Why this matters in your role:**  
Non-interoperable data sits in silos. A hospital trust with patient data in incompatible format to NHS England's systems can't share information for regional planning. A retailer with sales data incompatible with finance systems can't reconcile revenue.

---

### Understanding Interoperability

Interoperability enables data integration and combination. Requirements:

**Standard Formats**: Using widely adopted formats (CSV, JSON, XML) not custom formats.

**Common Vocabularies**: Using standardised terms and codes:
- ISO country codes (GB, not UK or United Kingdom)
- ISO date formats (2024-01-23, not 23/01/2024 or 01/23/2024)
- Industry-standard terminology

**Documented Schemas**: Clear specification of data structure, types, relationships.

<div class="alert alert-block alert-warning">
<b>Reminder:</b> <br>
Interoperability failures are expensive. Organisations spend millions converting incompatible data for integration. Using standards from the start prevents these costs.
</div>

### Worked Example: Checking Date Format Interoperability

Let's verify our data uses ISO 8601 standard date format:

In [ ]:
# Check date format compliance
sample_dates = df['time'].head()

print("DATE FORMAT ASSESSMENT")
print(f"\nSample dates:")
for date in sample_dates:
    print(f"  {date}")

# Check ISO 8601 compliance
iso_format = True
try:
    pd.to_datetime(df['time'], format='ISO8601')
    print(f"\n✓ ISO 8601 Compliant")
except:
    iso_format = False
    print(f"\n✗ Not ISO 8601 Compliant")

<div class="alert alert-block alert-info">
<b>What just happened?</b><br>

- We examined the date format used in our dataset
- We checked compliance with ISO 8601 international standard
- ISO 8601 format (YYYY-MM-DDTHH:MM:SS) is unambiguous across systems
- Non-standard dates (01/02/2024 - is this Jan 2 or Feb 1?) cause integration errors

**Key lesson:** Standard date formats prevent misinterpretation when combining datasets from different sources or countries.

</div>

---

## Part 5: Independent Task - FAIR Assessment Report

### Your Challenge

You're a data governance officer at a local council. The council collects weather data to support:  
- Gritting schedules (roads team)
- Flooding risk (emergency planning)
- Air quality monitoring (public health)

Multiple teams collect weather data separately, leading to duplication and inconsistencies. Your manager asks you to assess the weather dataset against FAIR principles and recommend improvements to enable data sharing across teams.

**Your requirements:**
1. Assess Findability (identifier, metadata, discoverability)
2. Assess Accessibility (protocols, authentication, formats)
3. Assess Interoperability (standards, vocabularies, schema documentation)
4. Assess Reusability (license, provenance, domain standards)
5. Provide scored assessment (0-100%) for each FAIR dimension
6. Recommend top 3 improvements to enable council-wide data sharing

**Getting started:**
- Use knowledge from Parts 2, 3, and 4
- Consider what makes data shareable across teams
- Think about barriers preventing current sharing

<div class="alert alert-block alert-success">
<b>Your turn:</b> <br>
Create a comprehensive FAIR assessment report for the weather dataset, including scores and recommendations.

<details>
    <summary style="color:black;font-weight:bold">Click here for a hint</summary>

Structure your report:
1. Findability score based on metadata completeness
2. Accessibility score based on format and retrieval methods
3. Interoperability score based on standards compliance
4. Reusability score based on documentation and licensing
5. Overall FAIR score (average)
6. Top 3 recommendations

Use clear headers and formatting.

</details>

</div>

<details>
    <summary style="color:green;font-weight:bold">Click here for the solution</summary>

```python
print("=== FAIR ASSESSMENT REPORT ===")
print("Dataset: London Weather Observations")
print("Context: Local Council Multi-team Data Sharing\n")

# Findability Assessment
print("1. FINDABILITY")
findability_criteria = {
    'Unique identifier': True,   # Dataset has clear name
    'Rich metadata': True,        # Creator, location, time range documented
    'Searchable repository': False,  # Not in catalogued system
    'Indexed': False              # Not discoverable via search
}
findability_score = (sum(findability_criteria.values()) / len(findability_criteria)) * 100
print(f"   Score: {findability_score:.0f}%")
print(f"   Status: {'✓ GOOD' if findability_score >= 75 else '⚠ NEEDS IMPROVEMENT'}\n")

# Accessibility Assessment
print("2. ACCESSIBILITY")
accessibility_criteria = {
    'Standard protocol': True,    # CSV retrievable via standard file access
    'Open format': True,          # CSV is open standard
    'Authentication defined': False,  # No access control policy
    'Metadata persistence': False # No retention policy documented
}
accessibility_score = (sum(accessibility_criteria.values()) / len(accessibility_criteria)) * 100
print(f"   Score: {accessibility_score:.0f}%")
print(f"   Status: {'✓ GOOD' if accessibility_score >= 75 else '⚠ NEEDS IMPROVEMENT'}\n")

# Interoperability Assessment
print("3. INTEROPERABILITY")
interoperability_criteria = {
    'Standard format': True,      # CSV is standard
    'ISO date format': True,      # ISO 8601 compliant
    'Common vocabularies': True,  # Standard units (°C, mph, hPa)
    'Schema documented': False    # No formal schema definition
}
interoperability_score = (sum(interoperability_criteria.values()) / len(interoperability_criteria)) * 100
print(f"   Score: {interoperability_score:.0f}%")
print(f"   Status: {'✓ GOOD' if interoperability_score >= 75 else '⚠ NEEDS IMPROVEMENT'}\n")

# Reusability Assessment
print("4. REUSABILITY")
reusability_criteria = {
    'License specified': False,   # No license documented
    'Provenance documented': True,  # Source is Open-Meteo API
    'Quality documented': False,  # No quality assessment documented
    'Usage guidelines': False     # No guidance for reuse
}
reusability_score = (sum(reusability_criteria.values()) / len(reusability_criteria)) * 100
print(f"   Score: {reusability_score:.0f}%")
print(f"   Status: {'✓ GOOD' if reusability_score >= 75 else '⚠ NEEDS IMPROVEMENT'}\n")

# Overall FAIR Score
overall_fair = (findability_score + accessibility_score + interoperability_score + reusability_score) / 4
print("=== OVERALL FAIR SCORE ===")
print(f"   {overall_fair:.0f}%")
if overall_fair >= 80:
    print("   ✓ EXCELLENT - Ready for council-wide sharing")
elif overall_fair >= 60:
    print("   ⚠ GOOD - Minor improvements needed")
else:
    print("   ✗ POOR - Significant improvements required")

# Recommendations
print("\n=== TOP 3 RECOMMENDATIONS ===")
print("1. Create data catalog entry")
print("   - Add dataset to council's data catalog system")
print("   - Enables teams to discover weather data without duplication")
print("   - Impact: Improves Findability from 50% to 100%\n")

print("2. Document formal schema and quality standards")
print("   - Create schema definition (variable names, types, units)")
print("   - Document quality assessment results")
print("   - Impact: Improves Interoperability and Reusability\n")

print("3. Define access policy and license")
print("   - Specify who can access for what purposes")
print("   - Add license terms (e.g., CC-BY for internal sharing)")
print("   - Create usage guidelines for council teams")
print("   - Impact: Improves Accessibility and Reusability\n")

print("Expected Outcome: Overall FAIR score improves to ~85%")
print("Benefit: All council teams can find, access, and reuse weather data")
```

</details>

In [ ]:
# Your FAIR assessment report here
# Include: F, A, I, R scores and top 3 recommendations



**Success criteria:**
- Score calculated for each FAIR dimension (F, A, I, R)
- Overall FAIR score computed
- Three specific, actionable recommendations
- Each recommendation tied to FAIR improvement
- Clear business benefit articulated

**Portfolio note:** FAIR assessment methodology applies to any organisational data. Use this framework for your Module Challenge Part A when evaluating workplace data management.

---

## Reflection

Reflect on the following two questions and record your thoughts in the cell below. Ground your answers in examples from your workplace, using **approximately 300 words** in total.

Your answers here form part of your Module Challenge Part A portfolio and are assessed against:
- **MLO1:** *Explain the roles and functions of technology solutions within an organisation*
- **MLO2:** *Critically reflect upon data management systems and their role in digital and technology solutions*

Where you draw on external sources, cite them in Harvard style. This does not count towards your word count.

Once you commit your notebook, your Coach will be able to provide feedback on your responses. You can develop your answers further before your final submission.

<div class="alert alert-block alert-info">

1. Which FAIR principle would most improve data sharing in your workplace? What barriers prevent implementing this principle? (MLO2)

2. What role do data management systems play in your workplace's digital and technology solutions? Using an example from your workplace, what are the strenths and challenges you find with the system? Justify your view with reference to one challenge you have encountered implementing, managing, or using it. (MLO1, MLO2)

</div>

*Write your responses here.*

1. 

2. 

References (if applicable):

---

## Summary

**You have completed Week 3 Section 2.** You can now:

- Explained the five data lifecycle stages (creation, storage, usage, archiving, deletion) and used pandas datetime calculations to categorise weather records by age and estimate archiving cost savings
- Assessed dataset findability by building a metadata dictionary and scoring metadata completeness against six discovery criteria
- Assessed data format accessibility (CSV as an open, non-proprietary standard) and designed access control policies for three different weather-data use cases
- Checked the dataset's date format against the ISO 8601 standard using `pd.to_datetime(..., format='ISO8601')`
- Built a complete FAIR assessment report for the weather dataset, scoring Findability, Accessibility, Interoperability, and Reusability, and recommending the top three improvements for council-wide data sharing

**Key takeaways:**

- Accessible data is not the same as open data. Accessible means retrievable by authorised users through documented methods. Open means available to anyone without restriction.
- The FAIR score is not a single measure. Each dimension (Findability, Accessibility, Interoperability, Reusability) is scored separately against its own criteria, then averaged into an overall figure.

**What comes next:**

- **Section 3** turns to combining datasets, covering appending and joining techniques, choosing the right join type for a business question, and building a data pipeline that extracts, transforms, and loads weather data automatically.

**Action items:**

1. Save this notebook (Ctrl+S or Cmd+S)
2. Commit and push your work to your GitHub repository with the commit message `"Complete Week 3 Section 2 Data Management and FAIR Principles"`
3. Make sure the repository shows your latest commit and that all files are up to date
4. Share a link to your repository with your Coach in the DT401.3 component on Aptem
5. Open `DT401_Week3_Section3.ipynb` when ready to continue